In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
with open('data/names.txt') as fobj:
    names = fobj.read().split('\n') # split file context at newline
# map letters to integer indices, including '.' (end-of-word character)
# this will help us build a transition matrix
alphabet = ".abcdefghijklmnopqrstuvwxyz"
idx = {letter:index for index, letter in enumerate(alphabet)}
idx

all_names = '.'+'.'.join(names)

Plan:
* Encode characters into one-hot vectors
* Stack together one-hot vectors into a window of a given length (here we'll use 3) to serve as the input data
* Train a NN with one hidden layer to predict next character

In [2]:
# set hyperparameters
window = 3
hidden_units = 100

# process data
# first convert letters to integer indices
all_idxs = torch.tensor([idx[c] for c in all_names])
# now convert to one-hot vectors
x_onehot = F.one_hot(all_idxs, num_classes=27).float()
x_onehot.shape # (num_samples, 27)

torch.Size([228146, 27])

In [4]:
x_onehot[:5], all_names[:5]

(tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
 '.emma')

In [5]:
# now stack together windows of letters (i.e. rows of x_onehot)
N = x_onehot.shape[0]
# create a tensor to hold the data
X = torch.zeros((N-window, window, 27))
for k in range(N-window):
    for j in range(window):
        X[k, j] = x_onehot[k+j]

# get the Y vector that's aligned with this X
# should have first row of Y = 4th character of the sequence (i.e. index 3)
Y = all_idxs[window:]

In [6]:
X.shape, Y.shape

(torch.Size([228143, 3, 27]), torch.Size([228143]))

In [7]:
X[0] # 3 x 27 array encoding 3 characters, '.em' 

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [8]:
Y[0] # single integer index encoding 'm' (the character following '.em')

tensor(13)

Construct the model:

* One hidden layer with `hidden_units` hidden units and a `tanh` nonlinearity
* Linear map to the output layer and a softmax activation
* Include bias at both layers

In [9]:
W1 = torch.randn((window * 27, hidden_units), requires_grad = True)
b1 = torch.randn((hidden_units,), requires_grad=True)

W2 = torch.randn((hidden_units, 27), requires_grad = True)
b2 = torch.randn((27,), requires_grad = True)

parameters = (W1, b1, W2, b2)

In [10]:
# need to reshape X so that it's (N, 81) rather than (N, 3, 27)
X.view(-1, 81).shape

torch.Size([228143, 81])

In [11]:
# run the model forward
hidden_activations = (X.view(-1, 81) @ W1 + b1).tanh() # tanh nonlinearity
# second layer
logits = hidden_activations @ W2 + b2

# compute the loss: same average negative log likelihood as we used last time
loss = F.cross_entropy(logits, Y)

In [13]:
loss.item()

13.612465858459473

In [21]:
# now let's train with gradient descent

h = .1 # step size
for _ in range(10):
    # clear the gradients
    for p in parameters:
        p.grad = None

    # run the model forward
    hidden_activations = (X.view(-1, 81) @ W1 + b1).tanh()
    logits = hidden_activations @ W2 + b2

    # compute loss
    loss = F.cross_entropy(logits, Y)
    print(loss.item())

    # gradient step
    loss.backward()
    for p in parameters:
        p.data -= h * p.grad    

3.982013702392578
3.978827953338623
3.975994110107422
3.9733903408050537
3.970940113067627
3.9685935974121094
3.966320276260376
3.9641001224517822
3.961919069290161
3.9597697257995605


In [46]:
# SGD
h = .001
for step in range(100):
    # clear the gradients
    for p in parameters:
        p.grad = None

    # grab a minibatch of size 64
    batch_idxs = torch.randint(0, X.shape[0], (64,)) # low, high, shape
    hidden_activations = (X[batch_idxs, :, :].view(-1, 81) @ W1 + b1).tanh()
    logits = hidden_activations @ W2 + b2

    loss = F.cross_entropy(logits, Y[batch_idxs])
    if step%100 ==0:
        print(loss.item())

    loss.backward()
    for p in parameters:
        p.data -= h * p.grad    

# much faster!!

2.9500908851623535


In [34]:
# sample from the model!

def embed(word):
    idxs = torch.tensor([idx[c] for c in word])
    xenc = F.one_hot(idxs, num_classes=27).float()
    return xenc.view(-1, 27*len(word))

def generate(starting_word = "..."):
    word = starting_word
    done = False

    while not done:
        # grab the window of appropriate length
        context = embed(word[-window:])

        # apply the NN function
        hidden_activations = (context @ W1 + b1).tanh()
        logits = hidden_activations @ W2 + b2

        # convert outputs to probabilities (subtract max to ensure numerical stability) Do we actually have to do this?
        next_char_probs = F.softmax(logits - logits.max(), dim=1)
        # sample the next letter according to probs
        next_idx = torch.multinomial(next_char_probs, 1)
        next_char = alphabet[next_idx]
        if next_char == '.':
            done = True
        word = word + next_char
    return word


In [50]:
for _ in range(5):
    print(generate('..e'))

..elinn.
..era.
..eraelesc.
..ereyn.
..erehala.


In [ ]:
# next: character embeddings